Langchain
  - 대규모 언어 모델을 활용한 혁신적인 프레임워크
  - 구성요소
    1. 랭체인 라이브러리 : 파이썬과 자바스크립트 라이브러리를 초함하여, 다양한 컴포넌트의 인터페이스와 통합. 이 컴포넌트들을 체인과 에이전트로 결합 할 수 있는 기본 런타임 그리고 체인과 에이전트의 사용 가능항 구현이 가능하다.
    2. 랭체인 템플릿 : 다양한 작업을 위한 쉽게 배포할 수 있는 아키텍처모음이다. 이 템플릿은 개발자들이 특정 작업 작업에 맞춰 빠르게 애플리케이션을 구축할 수 있다.
    3. 랭서브 : 랭체인 체인을 REST API로 배포할 수 있게 하는 라이브러리. 개발자들은 자신의 애플리케이션을 외부 시스템과 쉽게 통합할 수 있다.
    4. 랭스미스 : 개빌자 플랫폼으로 LLM프레임워크에서 구축된 체인을 디버깅, 테스트, 평가, 모니터링할 수 있으며, 랭체인과의 원할한 통합을 지원한다.

In [1]:
!pip install langchain>=0.3.0 langchain-openai langchain-community langchain-text-splitters

# 추가 유용한 패키지들
!pip install faiss-cpu tiktoken  # faiss(Facebook AI Similarity Search)-> 벡터DB

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 52.2 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = userdata.get('kosa')

In [3]:
# 설치확인

import langchain
from langchain_openai import ChatOpenAI

print(f'LangChain 버전 : {langchain.__version__}')

LangChain 버전 : 1.2.15


In [4]:
# 간단 테스트

try:
  llm = ChatOpenAI(model='gpt-4o-mini')
  response = llm.invoke("안녕~")
  print('설치완료')
  print(f'응답 : {response.content}')
except Exception as e:
  print(f'설정 오류 : {e}')

설치완료
응답 : 안녕하세요! 무슨 도움이 필요하신가요?


LLM 체인 만들기
  - LLM 체인(Prompt + LLM)은 LLM 기반 애플리케이션 개발에서 핵심적인 개념 중 하나이다.
  - 체인은 사용자의 입력(프롬프트)을 받아 LLM을 통해 적절한 응답이나 결과를 생성하는 구조이다

기본 LLM 체인의 구성요소
  - 프롬프트(Prompt) :
    1. 사용자 또는 시스템에서 제공하는 입력으로 LLM에게 특정 작업을 수행하도록 요청하는 지시문이다.
    2. 프롬프트는 질문, 명령, 문장 시작 부분 등 다양한 형태를 취할 수 있으며, LLM의 응답을 유도하는데 중요한 역할을 한다
  - LLM :    
    1. gpt, gemini등 대규모 언어 모델로 대량의 텍스트 데이터에서 학습하여 언어를 이해하고 생성할 수 있는 인공지능 시스템이다.
    2. LLM은 프롬프트를 바탕으로 적절한 응답을 생성하거나, 주어진  작업을 수행하는데 사용된다

작동방식
  - 프롬프트 생성
    1. 사용자의 요구 사항이나 특정 작업을 정의하는 프롬프트를 생성한다
    2. 이 프롬프트는 LLM에게 전달되기 전에 작업의 목적과 맥락을 명확히 전달하기 위해 최적화될 수 있다
  - LLM 처리
    1. LLM은 제공된 프롬프트를 분석하고, 학습된 지식을 바탕으로 적절히 응답을 생성한다
    2. 이 과정에서 LLM은 내부적으로 다양한 언어 패턴과 내외부 지식을 활용하여 요청된 작업을 수행하거나 정보를 제공한다.
  - 응답반환
    1. LLM에 의해 생성된 응답은 최종 사용자에게 필요한 형태로 변환되어 제공된다.
    2. 이 응답은 직접적인 답변, 생성된 텍스트, 요약된 정보 등 다양한 형태로 제공된다.

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini')

llm.invoke(input('질문 : '))

질문 : 너는 누구야? 


AIMessage(content='저는 AI 언어 모델인 챗GPT입니다. 다양한 질문에 답하고, 정보 제공이나 대화를 지원하기 위해 설계되었습니다. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 13, 'total_tokens': 51, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_50880c66bf', 'id': 'chatcmpl-DhDRYkyY4OZsvMo5cYs84pdNGTDRZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e401d-bde6-72b1-b548-1c12011e156a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 38, 'total_tokens': 51, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
result = llm.invoke('지구의 윤년에 주기는?')

result.content

'지구의 윤년 주기는 4년에 한 번입니다. 하지만 특정한 규칙이 있습니다. 기본적으로 4로 나누어 떨어지는 해는 윤년이지만, 100으로 나누어 떨어지는 해는 윤년이 아니고, 대신 400으로 나누어 떨어지는 해는 윤년입니다. 예를 들어, 2000년은 윤년이었지만 1900년은 윤년이 아니었습니다. 이 규칙은 태양년과 달력을 맞추기 위해 필요한 조정입니다.'

In [7]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model='gpt-4o-mini', max_tokens = 512)

answer = chatgpt.invoke(input('질문 : '))

print(answer.content)

질문 : pysical AI 에 대해 설명해줘 
물리적 AI(Physical AI)는 로봇 공학이나 물리적 시스템 내에서 인공지능을 적용하는 분야를 의미합니다. 이는 물체를 인식하고 상호작용하며, 특정 작업을 수행하는 기계를 설계하는 데 중점을 둡니다. 물리적 AI는 다양한 기술이 결합되어 있으며, 주로 다음과 같은 요소들을 포함합니다.

1. **로봇 공학**: 물리적 AI는 로봇을 통해 구현되며, 센서를 사용하여 주변 환경을 인식하고, 모터를 통해 물리적인 작업을 실행합니다.

2. **인식 및 감지**: 물리적 AI 시스템은 카메라, 라이다(Lidar), 초음파 센서 등 다양한 센서를 사용하여 주변 환경을 이해하고 물체를 인식합니다.

3. **제어 시스템**: AI 알고리즘은 로봇의 동작을 제어하고, 작업을 최적화하기 위한 의사결정을 지원합니다. 이는 일반적으로 기계 학습과 같은 기법을 포함합니다.

4. **청중 상호작용**: 물리적 AI는 인간과 상호작용할 수 있는 기능을 갖추고 있으며, 예를 들어 고객 서비스 로봇이나 교육용 로봇 등이 이에 해당합니다.

5. **자율성**: 물리적 AI 시스템은 특정 작업을 수행하기 위해 자율적으로 의사결정을 내리는 능력을 가질 수 있습니다. 이는 자율주행차나 드론에서 많이 볼 수 있습니다.

물리적 AI는 의료, 제조, 농업, 물류, 탐사 등의 다양한 분야에서 활용될 수 있으며, 사람과 협력하거나 인간의 능력을 보완하는 방식으로 지속적으로 발전하고 있습니다. 예를 들어, 물리적 AI를 적용한 로봇 수술 시스템은 수술의 정확성을 높이고 회복 시간을 단축하는 데 기여할 수 있습니다. 

이와 같은 기술들은 앞으로도 더욱 진화할 것으로 예상되며, 상호작용성과 자율성이 강화된 물리적 AI 시스템이 다양한 산업에서 사용될 것입니다.


Temperature 값 설정
  - 0이면 답변이 일관적
  - 1이면 답변이 매번 랜덤하게 바뀜(창의적인 답변)

In [8]:
chatgpt_temp0_1 = ChatOpenAI(model='gpt-4o-mini', temperature=0, max_tokens=512)
chatgpt_temp0_2 = ChatOpenAI(model='gpt-4o-mini', temperature=0, max_tokens=512)
chatgpt_temp1_1 = ChatOpenAI(model='gpt-4o-mini', temperature=1, max_tokens=512)
chatgpt_temp1_2 = ChatOpenAI(model='gpt-4o-mini', temperature=1, max_tokens=512)

model_list=[chatgpt_temp0_1, chatgpt_temp0_2, chatgpt_temp1_1, chatgpt_temp1_2]

for i in model_list:
  answer = i.invoke('왜 파이썬이 가장 인기있는 프로그래밍 언어야?')
  print('-'*100)
  print('>>> ', answer.content)

----------------------------------------------------------------------------------------------------
>>>  파이썬이 가장 인기 있는 프로그래밍 언어 중 하나인 이유는 여러 가지가 있습니다:

1. **간결하고 읽기 쉬운 문법**: 파이썬은 코드가 간결하고 명확하여, 초보자도 쉽게 배울 수 있습니다. 이는 코드의 가독성을 높이고 유지보수를 용이하게 합니다.

2. **다양한 용도**: 웹 개발, 데이터 분석, 인공지능, 머신러닝, 자동화 스크립트, 게임 개발 등 다양한 분야에서 사용됩니다. 이로 인해 많은 개발자들이 파이썬을 선택하게 됩니다.

3. **강력한 라이브러리와 프레임워크**: 파이썬은 NumPy, Pandas, Matplotlib, TensorFlow, Django, Flask 등 다양한 라이브러리와 프레임워크를 제공합니다. 이러한 도구들은 개발자들이 복잡한 작업을 쉽게 수행할 수 있도록 도와줍니다.

4. **활발한 커뮤니티**: 파이썬은 큰 사용자 커뮤니티를 가지고 있어, 문제 해결이나 정보 공유가 용이합니다. 많은 자료와 튜토리얼이 온라인에 존재하여 학습에 도움이 됩니다.

5. **다양한 플랫폼 지원**: 파이썬은 Windows, macOS, Linux 등 다양한 운영체제에서 사용할 수 있어, 플랫폼에 구애받지 않고 개발할 수 있습니다.

6. **기업의 채택**: 많은 대기업과 스타트업들이 파이썬을 사용하고 있으며, 이는 파이썬의 신뢰성과 안정성을 보여줍니다. 구글, 페이스북, NASA 등 다양한 기관에서 파이썬을 활용하고 있습니다.

이러한 이유들로 인해 파이썬은 많은 개발자와 기업에게 인기를 끌고 있으며, 지속적으로 성장하고 있는 언어입니다.
----------------------------------------------------------------------------------------------------
>>>  파이썬이 가장 인기 있는 프

ChatGPT처럼 실시간 응답 출력이 가능하도록 해보기

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 모델 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7
)

# stream() 사용
for chunk in llm.stream([
    HumanMessage(content="왜 파이썬이 가장 인기있는 언어야?")
]):
    print(chunk.content, end="", flush=True)

파이썬이 가장 인기 있는 프로그래밍 언어 중 하나인 이유는 여러 가지가 있습니다:

1. **쉬운 문법**: 파이썬은 문법이 간결하고 직관적이어서 초보자들이 배우기 쉽습니다. 코드가 읽기 쉽고 이해하기 쉬운 구조를 가지고 있어, 프로그래밍에 처음 입문하는 사람들에게 적합합니다.

2. **다양한 활용 분야**: 파이썬은 웹 개발, 데이터 분석, 인공지능, 머신러닝, 과학 컴퓨팅, 자동화 스크립트 등 다양한 분야에서 사용됩니다. 이러한 폭넓은 활용 가능성 덕분에 많은 개발자들이 선택하게 됩니다.

3. **강력한 라이브러리와 프레임워크**: 파이썬은 수많은 라이브러리와 프레임워크를 가지고 있어, 개발자들이 복잡한 기능을 쉽게 구현할 수 있도록 도와줍니다. 예를 들어, 데이터 과학에는 Pandas, NumPy, Matplotlib과 같은 라이브러리가 사용되고, 웹 개발에는 Django, Flask 등이 있습니다.

4. **활발한 커뮤니티**: 파이썬은 큰 커뮤니티를 가지고 있어, 문제를 해결하거나 새로운 정보를 얻는 데 용이합니다. 온라인 포럼, 문서, 튜토리얼 등이 풍부하여 학습과 문제 해결이 상대적으로 쉽습니다.

5. **다양한 플랫폼 지원**: 파이썬은 Windows, macOS, Linux 등 다양한 운영체제에서 사용할 수 있어 개발자들이 편리하게 작업할 수 있습니다.

6. **기업의 채택**: 많은 대기업들이 파이썬을 사용하고 있으며, 이는 신뢰성과 안정성을 높여줍니다. 구글, 페이스북, NASA 등 여러 유명 기업에서 파이썬을 사용하고 있습니다.

이러한 이유들 덕분에 파이썬은 많은 사람들에게 인기를 얻고 있으며, 지속적으로 성장하고 있습니다.

CallbackHandler 시스템
  - 콜백시스템은 LLM 애플리케이션의 다양한 단계를 모니터링할 수 있게 해주는 강력한 기능
  - 로깅, 모니터링, 스트리밍및 기타 작업에 유용하다

In [11]:
import time
time.time()

1779192503.5087972

In [13]:
# 표준 라이브러리
import time  # 실행 시간 측정
from datetime import datetime  # 타임스탬프 기록
from typing import Dict, Any  # 타입 힌팅

# start_time = time.time()

# LangChain 관련
from langchain_core.callbacks.base import BaseCallbackHandler  # 콜백 기본 클래스
from langchain_core.outputs import LLMResult  # LLM 결과 타입
from langchain_core.prompts import ChatPromptTemplate  # 프롬프트 템플릿
from langchain_openai import ChatOpenAI  # OpenAI 챗봇 모델

# 1. AstronomyCallbackHandler

class AstronomyCallbackHandler(BaseCallbackHandler):
  # 초기화
  def __init__(self, verbose: bool = True):
    self.verbose = verbose # 상세 로그 출력 여부
    self.start_time = None # 시작 시간 기록
    self.chain_steps = []  # 실행 단계 저장 리스트
    self.token_usage = {   # 토큰 사용량 추정
        'prompt_tokens':0,     # 입력 토큰
        'completion_tokens':0, # 출력 토큰
        'total_tokens':0       # 전체 토큰
    }

  # on_chain_start (체인 시작시)
  # 랭체인 실행 시작시 호출
  def on_chain_start(self, serialized: Dict[str, Any], inputs: Dict[str, Any], **kwargs)-> None:
    self.start_time= time.time()  # 시작 시간 기록
    self.chain_steps = []         # 단계 초기화
    if self.verbose:  # 상세모드면
      print(f'[체인 시작] {datetime.now().strftime("%H:%M:%S")}')
      print(f'입력 : {inputs}')
      print('-'*50)

  # on_llm_end (LLM 호출 완료시)
  def on_llm_end(self, response: LLMResult, **kwargs)-> None:
    self.chain_steps.append('LLM 호출 완료')  # 단계 기록

    # 토큰 사용량 누적
    if hasattr(response, 'llm_output') and response.llm_output:
      usage = response.llm_output.get('token_usage', {})
      self.token_usage['prompt_tokens'] += usage.get('prompt_tokens', 0)
      self.token_usage['completion_tokens'] += usage.get('completion_tokens', 0)
      self.token_usage['total_tokens'] += usage.get('total_tokens', 0)

# 2. PerformanceCallbackHandler
# >> 성능 측정에 특화된 콜백 핸들러 (실행시간 측정에 집중)

class PerformanceCallbackHandler(BaseCallbackHandler):
  def __init__(self):
    self.performance_data={}     # 성능 데이터 저장할 딕셔너리 {단계명: 소요시간}
    self.current_step= None      # 현재 실행중인 단계
    self.step_start_time= None   # 각 단계의 시작 시간

  def on_llm_start(self, serialized, prompts, **kwargs):
    self.current_step = 'llm_processing'  # 현재 단계 설정
    self.step_start_time = time.time()    # 타이머 시작

  def on_llm_end(self, response, **kwargs):
    if self.current_step and self.step_start_time:
      duration = time.time() - self.step_start_time       # 경과시간 계산
      self.performance_data[self.current_step] = duration # 저장

  # 성능 리포트 조회 {'llm_processing': 2.347} (2.347초 소요)
  def get_report(self):
    return self.performance_data

# 3. LoggingCallbackHandler
# 로깅 전용 콜백 핸들러 >> 실행과정 기록 및 파일 저장

class LoggingCallbackHandler(BaseCallbackHandler):

  def __init__(self, log_file: str= None):
    self.log_file = log_file   # 로그 파일 경로
    self.logs= []              # 메모리에 저장할 로그 리스트

  # 내부 로깅 메소드
  def _log(self, message: str):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S') # 타임스탬프 생성
    log_entry = f'[{timestamp}] {message}'  # 로그 엔트리 포맷
    self.logs.append(log_entry)             # 메모리에 저장

    if self.log_file:  # 파일 경로 지정되어 있다면
      with open(self.log_file, 'a', encoding='utf-8') as f:
        f.write(log_entry + "\n") # 파일에 추가

  # on_llm_start: llm 시작시
  def on_llm_start(self, serialized, prompts, **kwargs):
    self._log(f'LLM 시작 - 프롬프트 : {prompts}')

  # on_llm_end: llm 완료시
  def on_llm_end(self, response, **kwargs):
    self._log(f'LLM 완료 - 응답 길이 : {len(str(response))}')


# 콜백 인스턴스 생성
performance_callback = PerformanceCallbackHandler() # 성능 측정
logging_callback = LoggingCallbackHandler()         # 로그 기록
astronomy_callback = AstronomyCallbackHandler()     # 토큰 추적

# 리스트로 묶어서 관리
callbacks = [performance_callback, logging_callback, astronomy_callback]

# 랭체인 구성(LLM + 프롬프트체인 구성)

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template('천문학 주제에 대해 설명해줘 : {topic}')
# 모델 설정
llm = ChatOpenAI(model='gpt-4o')
# 체인 연결: 프롬프트 >> llm
chain = prompt | llm

# 테스트 실행

test_topics = ['블랙홀', '중성자별', '적색거성']

for topic in test_topics:
  result = chain.invoke({'topic' : topic},              # 입력데이터
                        config={'callbacks':callbacks}) # 콜백 핸들러 전달


  print(f'\n 결과({topic})\n{result.content if hasattr(result, "content") else result}')

# 성능 지표출력
print('\n 성능 지표 : ')
perf_report = performance_callback.get_report()
for metric, value in perf_report.items():
  print(f' - {metric} : {value :.3f}초')

# 토큰 사용량
print('\n 토큰 사용량 : ')
usage = astronomy_callback.token_usage
print(f' - Prompt Tokens : {usage["prompt_tokens"]}')
print(f' - Completion Tokens : {usage["completion_tokens"]}')
print(f' - Total Tokens : {usage["total_tokens"]}')

# 비용 계산 (매우 실용적)
# 모델별 단가 테이블
MODEL_PRICE = {
    'gpt-4o':{'input':0.005, 'output':0.015},
    'gpt-4o-mini':{'input':0.003, 'output':0.006},
    'gpt-3.5-turbo':{'input':0.001, 'output':0.002},
}

# 과금 계산

model_name = 'gpt-4o'
if model_name in MODEL_PRICE:
  price_in = MODEL_PRICE[model_name]['input']    # 입력 단가
  price_out = MODEL_PRICE[model_name]['output']  # 출력 단가

  cost_input = (usage['prompt_tokens'] / 1000) * price_in
  cost_output = (usage['completion_tokens'] / 1000) * price_out
  total_cost = cost_input + cost_output

  print(f'\n [모델 : {model_name}]')
  print(f' - 입력단가 : ${price_in:.4f} / 1k tokens')
  print(f' - 출력단가 : ${price_out:.4f} / 1k tokens')
  print(f' - 입력요금 : ${cost_input:.6f}')
  print(f' - 출력요금 : ${cost_output:.6f}')
  print(f' - 총 예상 요금 : ${total_cost:.6f}')



[체인 시작] 12:25:05
입력 : {'topic': '블랙홀'}
--------------------------------------------------
[체인 시작] 12:25:05
입력 : {'topic': '블랙홀'}
--------------------------------------------------

 결과(블랙홀)
블랙홀은 일반 상대성이론에 의해 예측된, 중력이 극도로 강력하여 어떤 것도, 심지어 빛조차도 빠져나갈 수 없는 시공간의 한 영역입니다. 블랙홀은 주로 매우 무거운 별이 자신의 중력을 이기지 못하고 붕괴할 때 형성됩니다. 이처럼 붕괴된 별의 중심에는 특이점(singularity)이 존재하며, 이 특이점은 무한한 밀도와 중력을 가진다고 여겨집니다.

블랙홀의 경계는 사건의 지평선(event horizon)이라고 불리며, 이 경계를 넘어선 것은 다시는 그 안에서 나올 수 없습니다. 사건의 지평선은 블랙홀의 크기를 정의하는 부분이며, 그 크기는 블랙홀의 질량에 비례합니다.

블랙홀은 종류에 따라 크기와 질량이 다양합니다. 가장 일반적인 종류는 항성 질량 블랙홀(stellar black hole)과 초대질량 블랙홀(supermassive black hole)입니다. 항성 질량 블랙홀은 태양의 몇 배에서 수십 배 정도의 질량을 가지며, 초대질량 블랙홀은 수백만에서 수십억 배의 태양 질량을 가지고 있으며, 보통 은하의 중심에 위치합니다.

블랙홀은 주변 물질을 강하게 끌어당기며, 이 물질이 블랙홀로 빨려 들어가는 과정에서 생기는 마찰과 압축 때문에 엄청난 에너지를 방출합니다. 이로 인해 발생하는 현상 중 하나가 강력한 엑스선 방출이며, 이러한 방출은 블랙홀을 간접적으로 탐지하게 해줍니다. 블랙홀이 형성되는 과정, 물리적 특성 및 그 주위에서 일어나는 현상들은 현대 천문학 및 이론물리학에서 중요한 연구 분야로 남아 있습니다.
[체인 시작] 12:25:14
입력 : {'topic': '중성자별'}
-------------------------

행 흐름 다이어그램
```
사용자 입력: "블랙홀"
    ↓
[프롬프트 템플릿] "천문학 주제에 대해 설명해줘 : 블랙홀"
    ↓
[콜백 시작]
├─ AstronomyCallbackHandler.on_chain_start()
├─ LoggingCallbackHandler.on_llm_start()
└─ PerformanceCallbackHandler.on_llm_start()
    ↓
[GPT-4o 처리]
    ↓
[콜백 종료]
├─ AstronomyCallbackHandler.on_llm_end() → 토큰 누적
├─ LoggingCallbackHandler.on_llm_end() → 로그 기록
└─ PerformanceCallbackHandler.on_llm_end() → 시간 기록
    ↓
결과 출력 + 메트릭 계산